<a href="https://colab.research.google.com/github/ikabrain/UCS772-CV-NLP-Lab/blob/main/NLP_assign5/NLP_assign5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 5 - Sentiment Analysis using advanced methods
---

In [1]:
import numpy as np
import pandas as pd
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence

SEED = 42

In [2]:
# Load keras's pre-processed IMDb Movie Review Dataset
NUM_WORDS = 10000
PAD, START, UNK, INDEX_FROM = 0, 1, 2, 3 # !IMPORTANT: shift is 0 = pad, 1 = start, 2 = unknown, real words start at 3

(X_train, y_train), (X_test, y_test) = imdb.load_data(
    num_words=NUM_WORDS,
    start_char=START,
    oov_char=UNK,
    index_from=INDEX_FROM,
    seed=SEED
)

print("Training samples: ", len(X_train))
print("Testing samples: ", len(X_test))

print("Sentiment values: ", np.unique(y_train))

Training samples:  25000
Testing samples:  25000
Sentiment values:  [0 1]


In [3]:
MAX_LEN = 200

X_train = sequence.pad_sequences(X_train, maxlen=MAX_LEN, padding="pre", truncating="pre", value=PAD)
X_test = sequence.pad_sequences(X_test, maxlen=MAX_LEN, padding="pre", truncating="pre", value=PAD)

print("Training shape: ", X_train.shape)
print("Testing shape: ", X_test.shape)

Training shape:  (25000, 200)
Testing shape:  (25000, 200)


In [4]:
# Decoding one review back to words
word_index = imdb.get_word_index()

inv = {v + INDEX_FROM: k for k, v in word_index.items()}
inv.update({PAD: "<pad>", START: "<start>", UNK: "<unk>"})

print("First review:", " ".join(inv.get(t, "<unk>") for t in X_train[0] if t != PAD))
print("Its label:", y_train[0])

First review: plays the same kind of role in the <unk> bogart film the <unk> sadly i can kind of identify with mostel in that last chase scene where he and palance are being chased down by widmark douglas and half the new orleans police seeing the weight challenged zero trying to keep up with palance was something else because i'm kind of in <unk> league now in the <unk> department br br kazan kept the action going at a good clip there's very little down time in this film if there was any less it would be an indiana jones film panic in the streets won an oscar for best original screenplay that year br br kazan also made good use of the new orleans <unk> and the french quarter some of the same kinds of shots are later used in on the <unk> in fact panic in the streets is about people not <unk> when they really should in their own best interest very similar again to on the <unk> br br panic in the streets does everyone proud who was associated with it now why couldn't <unk> kazan get some

## Q1: Applying RNN & LSTM for Sentiment Analysis
---

Apply deep learning models i.e. RNN and LSTM for text classification for sentiment
analysis.

In [5]:
BATCH_SIZE = 64
EPOCHS = 2

In [6]:
# Loading pre-trained GloVe Embeddings
!pip install -q gensim

import gensim.downloader as api

kv = api.load("glove-wiki-gigaword-100")
print("No of KeyedVector: ", len(kv))
print("Size of each KeyedVector: ", kv.vector_size)

No of KeyedVector:  400000
Size of each KeyedVector:  100


In [7]:
# Building Embedding matrix to work with integer representation of words

rng = np.random.default_rng(SEED)
E = rng.normal(0, 0.1, (NUM_WORDS, kv.vector_size)).astype(np.float32)
E[PAD] = 0

hits = 0
for word, rank in word_index.items():
    i = rank + INDEX_FROM
    if i < NUM_WORDS and word in kv:
        E[i] = kv[word]
        hits += 1

print(f"GloVe coverage: {hits / (NUM_WORDS - INDEX_FROM):.1%}")

GloVe coverage: 98.0%


### Designing Base Classes

In [8]:
class SequenceCell:
    """
    Base class for the Vanilla and LSTM cells that compute forward and backward per timestep.
    """
    def init_state(self):
        # Initialsing hidden (and cell) state
        raise NotImplementedError

    def forward(self, x_t, prev_state):
        raise NotImplementedError # returns (new_state, cache)

    def __call__(self, x_t, prev_state):
        return self.forward(x_t, prev_state)

    def backward(self, dstate, cache):
        raise NotImplementedError # returns (dx_t, dprev_state); accumulates self.grads

    def _init_zero_grads(self):
        return {k: np.zeros_like(v) for k, v in self.params.items()} # assign to self.grads for grad wrt each param

    def zero_dstate(self):
        return tuple(np.zeros_like(s) for s in self.init_state()) # returns initial zero grad of loss wrt states; accumulates dstate in recurrent layers

In [9]:
class Layer:
    """
    Base class for Recurrent and Activation layers of the neural network.
    """
    def forward(self, X):
        raise NotImplementedError

    def __call__(self, X):
        return self.forward(X)

    def backward(self, dA):
        # Here, dA is the activated gradient from the NEXT layer that has propagated to here for current output!
        raise NotImplementedError

In [10]:
class Loss:
    """
    Base class for loss function.
    """
    def forward(self, y_true, y_pred):
        raise NotImplementedError

    def __call__(self, y_true, y_pred):
        return self.forward(y_true, y_pred)

    def backward(self):
        # Returns gradient of loss function
        raise NotImplementedError

In [ ]:
class Optimizer:
    """
    Base class for optimizers.
    """
    def update(self, params: dict[str, np.ndarray], grads: dict[str, np.ndarray]):
        raise NotImplementedError

In [11]:
class Model:
    """
    Base class for model as combination of layers.
    """
    def __init__(self, layers = []):
        self.layers = layers
        self.loss = None

    def add(self, layer):
        self.layers.append(layer)

    def forward(self, X):
        for layer in self.layers:
            X = layer(X)
        return X

    def __call__(self, X):
        return self.forward(X)

    def backward(self, dA):
        for layer in reversed(self.layers):
            dA = layer.backward(dA)

    def train(self, X, y, loss_fn: Loss, optimizer, epochs):
        return NotImplementedError

### Implementing RNN from scratch

In [28]:
class RNNCell(SequenceCell):
    def __init__(self, input_dim, n_neurons, seed=42) -> None:
        """
        input_dim = Dimensionality of the input vector
        n_neurons = Number of neurons in the hidden state layer
        """
        self.x_dim = input_dim
        self.h_dim = n_neurons

        rng = np.random.default_rng(seed)
        # Assuming ROW vectors as inputs i.e. transposed weights
        self.params = {
            "Whh": 0.1*rng.standard_normal(size=(self.h_dim, self.h_dim)),
            "Wxh": 0.1*rng.standard_normal(size=(self.x_dim, self.h_dim)),
            "bh": 0.1*rng.standard_normal(size=(self.h_dim,))
        }

        self.grads = self._init_zero_grads()

    def forward(self, x_t, prev_h):
        logit = x_t @ self.params["Wxh"] + prev_h @ self.params["Whh"] + self.params["bh"]
        h = np.tanh(logit)

        new_state = (h,)
        cache = (x_t, prev_h, h)

        return (new_state, cache)

    def backward(self, dstate, cache):
        (dh,) = dstate
        (x_t, prev_h, h) = cache
        dpre = dh * (1 - h**2) # Gradient of tanh

        self.grads["Wxh"] += x_t.T @ dpre
        self.grads["Whh"] += prev_h.T @ dpre
        self.grads["bh"] += dpre

        dx_t = dpre @ self.params["Wxh"].T
        dprev_h = dpre @ self.params["Whh"].T

        dprev_state = (dprev_h,)
        return (dx_t, dprev_state)

In [25]:
# TODO: Implement Recurrent Layer
class RNNLayer(Layer):
    self.T = max(X_t.shape)
    self.H = np.zeros((self.T+1, hidden_size)) # Here, H[t] = h_t, and h_0 exists so T+1 needed!
    # TODO: return_sequences gives me h for all times, so I can use it for not just sentiment classification

In [ ]:
class Sigmoid(Layer):
    """
    Activation function applied to final hidden state output.
    """
    def forward(self, Z):
        self.Z = Z
        self.out = 1 / (1 + np.exp(-self.Z))
        return self.out

    def backward(self, dA):
        return dA * self.out * (1-self.out) # Derivative of sigmoid is f(1-f)

In [ ]:
# TODO: Implement Loss

In [ ]:
# Implementing Adam from scratch

class AdamOptimizer:
    """
    Adam optimizer implementation.
    For RNNs, used for updating shared params AFTER all the gradients have propagated through ALL times!
    """
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        """
        Initializes the Adam optimizer parameters.
        """
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon

        # State tracking dictionaries for parameters
        self.m = {}
        self.v = {}
        self.t = 0

    def initialize_params(self, params):
        """
        Initializes moment vectors to zero shapes matching the model parameters.
        """
        for key, val in params.items():
            self.m[key] = np.zeros_like(val)
            self.v[key] = np.zeros_like(val)

    def update(self, params, grads):
        """
        Updates parameters in-place using computed gradients.

        params: dict of parameter arrays (e.g., {"W1": W1, "b1": b1})
        grads: dict of gradient arrays corresponding to params (e.g., {"W1": dW1, "b1": db1})
        """
        # Lazy initialization if not done yet
        if not self.m:
            self.initialize_params(params)

        self.t += 1

        for key in params.keys():
            # 1. Update biased first moment estimate
            self.m[key] = self.beta1 * self.m[key] + (1 - self.beta1) * grads[key]

            # 2. Update biased second moment estimate
            self.v[key] = self.beta2 * self.v[key] + (1 - self.beta2) * (grads[key] ** 2)

            # 3. Compute bias-corrected first moment estimate
            m_hat = self.m[key] / (1 - self.beta1 ** self.t)

            # 4. Compute bias-corrected second moment estimate
            v_hat = self.v[key] / (1 - self.beta2 ** self.t)

            # 5. Update parameters
            params[key] -= (self.lr / (np.sqrt(v_hat) + self.epsilon)) * m_hat

        return params

In [5]:
# TODO: Implement sequential SentimentModel class (must apply a given Embedding matrix to the provided input)

### Applying custom RNN for sentiment analysis

### Implementing LSTM from scratch

### Applying custom LSTM for sentiment analysis

## Q2: Comparing models
---

Compare both models based on various evaluation metrics using your results.

## Review
---

Answer the following questions based on your experimental results:
<ol type="a">
    <li>Which model performed better and why?</li>
    <li>Draw suitable visualizations for the results using both models to make the experimental results more interpretable and write your observations.</li>
    <li>Compare based on trainable parameters.</li>
</ol>